In [3]:
!pip -q install -U huggingface_hub

from huggingface_hub import login
login()  # will prompt for a token


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 533.4/533.4 kB 33.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.57.3 requires huggingface-hub<1.0,>=0.34.0, but you have huggingface-hub 1.3.1 which is incompatible.
Error importing huggingface_hub._login: cannot import name 'ANSI' from 'huggingface_hub.utils' (/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/__init__.py)


ImportError: cannot import name 'ANSI' from 'huggingface_hub.utils' (/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/__init__.py)

In [4]:
# ============================================================
# COLAB CELL — Load 3 student families (base + instruct) and run inference on 200 samples
# - Uses EXACT row["prompt"] as-is (NO system/user wrapping, NO extra tokens you add manually)
# - For instruct models: if tokenizer has a chat_template, we use it with a SINGLE user message = prompt text
#   (still no additional content; just packaging). If no template exists, we fall back to raw prompt text.
# - Saves outputs to a single JSON
# ============================================================

!pip -q install -U transformers accelerate sentencepiece
from google.colab import drive
drive.mount("/content/drive")
import os, json, random, time
from typing import Dict, List

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# ----------------------------
# CONFIG (filled with your families)
# ----------------------------

TEST_JSONL = "/content/drive/MyDrive/DL_Local/clinical_pharm_prompts_10000.jsonl"
OUT_JSON   = "/content/drive/MyDrive/DL_Local/student_inference_200.json"

N_SAMPLES = 200
SEED = 42

# 3 families × (base, instruct) = 6 entries
STUDENTS = {
    # Qwen2.5 1.5B
    "qwen25_1p5b_base":      {"path": "Qwen/Qwen2.5-1.5B",           "is_instruct": False},
    "qwen25_1p5b_instruct":  {"path": "Qwen/Qwen2.5-1.5B-Instruct",  "is_instruct": True},

    # Qwen2.5 3B
    "qwen25_3b_base":        {"path": "Qwen/Qwen2.5-3B",             "is_instruct": False},
    "qwen25_3b_instruct":    {"path": "Qwen/Qwen2.5-3B-Instruct",    "is_instruct": True},

    # Qwen2.5 7B
    "qwen25_7b_base":        {"path": "Qwen/Qwen2.5-7B",             "is_instruct": False},
    "qwen25_7b_instruct":    {"path": "Qwen/Qwen2.5-7B-Instruct",    "is_instruct": True},
}


# Generation params
GEN_KW = dict(
    max_new_tokens=2000,
    do_sample=False,
    temperature=0.0,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32
print("DEVICE:", DEVICE, "DTYPE:", DTYPE)

# ----------------------------
# Helpers
# ----------------------------

def load_jsonl(path: str) -> List[dict]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows

def pick_samples(rows: List[dict], n: int, seed: int) -> List[dict]:
    rng = random.Random(seed)
    if n >= len(rows):
        return rows[:]
    idx = list(range(len(rows)))
    rng.shuffle(idx)
    return [rows[i] for i in idx[:n]]

def extract_prompt_text(row: dict) -> str:
    # You said: field "prompt" has everything needed.
    if "prompt" in row and isinstance(row["prompt"], str):
        return row["prompt"]
    raise KeyError(f'Expected key "prompt" in row. keys={list(row.keys())}')

def format_for_model(tokenizer, prompt_text: str, is_instruct: bool) -> str:
    """
    IMPORTANT: No added content.
    - Base: return raw prompt_text.
    - Instruct: if tokenizer has chat_template, wrap as a SINGLE user message whose content is prompt_text.
               Otherwise, return raw prompt_text.
    """
    if not is_instruct:
        return prompt_text

    if hasattr(tokenizer, "apply_chat_template") and getattr(tokenizer, "chat_template", None):
        msgs = [{"role": "user", "content": prompt_text}]
        return tokenizer.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=True
        )

    # If no template exists, do not invent formatting; just pass raw prompt.
    return prompt_text

@torch.inference_mode()
def generate_one(model, tokenizer, text: str) -> str:
    inputs = tokenizer(text, return_tensors="pt", truncation=True)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    out = model.generate(**inputs, **GEN_KW)
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    return decoded

def postprocess_to_answer_only(full_decoded: str, model_input: str) -> str:
    # Heuristic: strip echoed prompt if present.
    if full_decoded.startswith(model_input):
        return full_decoded[len(model_input):].lstrip()
    return full_decoded.strip()

# ----------------------------
# Load prompts
# ----------------------------
rows = load_jsonl(TEST_JSONL)
samples = pick_samples(rows, N_SAMPLES, SEED)

print(f"Loaded rows: {len(rows)} | Using samples: {len(samples)}")
print("Example keys:", list(samples[0].keys()))

prompt_pack = []
for r in samples:
    rid = r.get("id") or r.get("ID") or r.get("_id") or r.get("uid") or None
    prompt_pack.append({"id": rid, "prompt": extract_prompt_text(r)})

print("First prompt id:", prompt_pack[0]["id"])

# ----------------------------
# Load models
# ----------------------------
models = {}
tokenizers = {}

for name, cfg in STUDENTS.items():
    path = cfg["path"]
    print(f"\nLoading {name} from {path} ...")

    tok = AutoTokenizer.from_pretrained(path, use_fast=True, trust_remote_code=True)

    # Ensure pad token exists
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        path,
        torch_dtype=DTYPE,
        device_map="auto" if DEVICE == "cuda" else None,
        low_cpu_mem_usage=True,
        trust_remote_code=True
    )
    model.eval()

    tokenizers[name] = tok
    models[name] = model

print("\n✅ All student models loaded:", list(models.keys()))

# ----------------------------
# Inference loop
# ----------------------------
results = {
    "meta": {
        "test_jsonl": TEST_JSONL,
        "n_samples": len(prompt_pack),
        "seed": SEED,
        "gen": GEN_KW,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    },
    "models": {name: {"path": cfg["path"], "is_instruct": cfg["is_instruct"]} for name, cfg in STUDENTS.items()},
    "samples": []
}

for i, item in enumerate(prompt_pack, 1):
    pid = item["id"]
    prompt_text = item["prompt"]

    sample_out = {"id": pid, "prompt": prompt_text, "outputs": {}}

    for mname, cfg in STUDENTS.items():
        tok = tokenizers[mname]
        model = models[mname]

        model_input = format_for_model(tok, prompt_text, is_instruct=cfg["is_instruct"])
        decoded = generate_one(model, tok, model_input)
        answer = postprocess_to_answer_only(decoded, model_input)

        sample_out["outputs"][mname] = {
            "raw_decoded": decoded,
            "answer": answer,
        }

    results["samples"].append(sample_out)

    if i % 10 == 0:
        print(f"Done {i}/{len(prompt_pack)}")

# ----------------------------
# Save JSON
# ----------------------------
with open(OUT_JSON, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"\n✅ Saved outputs JSON to: {OUT_JSON}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 39.1 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DEVICE: cuda DTYPE: torch.float16
Loaded rows: 10000 | Using samples: 200
Example keys: ['id', 'risk_bucket', 'age', 'sex', 'weight_kg', 'height_cm', 'bmi', 'renal_function', 'liver_function', 'medical_history', 'current_meds', 'new_med', 'prompt']
First prompt id: 1e0d9a7340bf

Loading qwen25_1p5b_base from Qwen/Qwen2.5-1.5B ...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Done 180/200


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for

Done 190/200


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for

Done 200/200

✅ Saved outputs JSON to: /content/drive/MyDrive/DL_Local/student_inference_200.json


In [5]:
# ============================================================
# COLAB — Unassign runtime (official API)
# ============================================================

from google.colab import runtime

print("✅ Job finished. Unassigning runtime...")
runtime.unassign()


✅ Job finished. Unassigning runtime...
